# Klasifikasi DemogPairs Menggunakan ViT (Wajah) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-face.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_rf_vit-face_',
    results_path='results/demogpairs_rf_vit-face_'
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}


Accuracy  : 0.8546296296296296
Precision : 0.8542723843783708
Recall    : 0.8546296296296297
F1 Score  : 0.8538912669595423
               precision    recall  f1-score   support

Asian_Females     0.8235    0.8167    0.8201       360
  Asian_Males     0.8274    0.8389    0.8331       360
Black_Females     0.8514    0.7639    0.8053       360
  Black_Males     0.8716    0.8861    0.8788       360
White_Females     0.8774    0.8944    0.8858       360
  White_Males     0.8743    0.9278    0.9003       360

     accuracy                         0.8546      2160
    macro avg     0.8543    0.8546    0.8539      2160
 weighted avg     0.8543    0.8546    0.8539      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9402777777777778,0.8235294117647058,0.8166666666666667,0.8200836820083681,360
Asian_Males,0.9439814814814815,0.8273972602739726,0.8388888888888889,0.8331034482758621,360
Black_Females,0.9384259259259259,0.8513931888544891,0.7638888888888888,0.8052708638360174,360
Black_Males,0.9592592592592593,0.8715846994535519,0.8861111111111111,0.8787878787878788,360
White_Females,0.961574074074074,0.8773841961852861,0.8944444444444445,0.8858321870701513,360
White_Males,0.9657407407407408,0.8743455497382199,0.9277777777777778,0.9002695417789757,360


Confusion matrix saved: images\cm_rf_vit-face_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               294                28                23                 0                13                 2
         Asian_Males                25               302                 2                 8                 2                21
       Black_Females                18                11               275                31                23                 2
         Black_Males                 0                11                11               319                 0                19
       White_Females                19                 2                11                 2               322                 4
         White_Males                 1                11                 1                 6                 7               334


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-face_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8546296296296296,0.8538912669595423,0.8542723843783708,0.8546296296296297,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-face_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 3667.0,
 'days': 0,
 'hours': 1,
 'minutes': 1,
 'seconds': 7.0,
 'text': '0 hari 1 jam 1 menit 7.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 14111.0,
 'days': 0,
 'hours': 3,
 'minutes': 55,
 'seconds': 11.0,
 'text': '0 hari 3 jam 55 menit 11.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8825,0.8681,0.8652,0.8681,0.8698,0.8707,0.8702,0.8713,0.8707,8.2339
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8825,0.8681,0.8663,0.8704,0.8663,0.8707,0.8703,0.8712,0.8707,9.5206
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8854,0.8669,0.8652,0.8721,0.8634,0.8706,0.8702,0.8712,0.8706,9.1365
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8848,0.8675,0.8652,0.8709,0.8634,0.8704,0.87,0.871,0.8704,8.8774
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.8767,0.86,0.8524,0.8605,0.8565,0.8612,0.8603,0.8617,0.8612,6.0663
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.8767,0.86,0.8524,0.86,0.8565,0.8611,0.8602,0.8615,0.8611,6.8587
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.8767,0.8588,0.8524,0.8605,0.8565,0.861,0.8601,0.8614,0.861,6.1877
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.8767,0.8588,0.8524,0.86,0.8565,0.8609,0.86,0.8613,0.8609,6.944
